# 📘 MODULE 5 · Assignment M05.4: Categorical Inference (15 pts)
**Due in Module 5.** When you finish, upload your PDF to the Canvas assignment named **M05.4**.

**This week you:** test whether two categorical variables are associated (chi-square), in service of YOUR research question.

**Same 5 steps as every week:** 1 run setup → 2 fill in your project card → 3 load data → 4 do this week's TODO cells → 5 run the submission block and export the PDF.

**You're done when:** hypothesis and variables pass self-checks, the test output is visible, and the 📤 submission block prints cleanly.

🧰 **First time here?** You will not write any Python. Press ▶ on a cell to run it, and type words between the quotation marks in 🟨 TODO cells. 🟥 Red error text can't hurt anything. 🟨 Yellow "NOT DONE YET" is coaching, not an error. `Runtime → Run all` fixes most weird problems. Stuck for 15 minutes? Screenshot it and message me.

**Before anything else:** `File → Save a copy in Drive`, then rename it `HIM4331_M05.4_LastnameFirstname`.

## ▶ Step 1: setup (nothing to type, just press ▶)

In [ ]:
#@title ▶ Step 1: tools and self-check helpers (just run it)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import datetime

pd.set_option("display.max_columns", 60)

_CHECKS = {}   # every self-check records its latest result here; the submission block reports them

def check(condition, ok_msg, fix_msg):
    """Friendly self-check: green check if done, yellow hint if not."""
    _CHECKS[fix_msg] = (bool(condition), ok_msg)
    if condition:
        print("✅ " + ok_msg)
        return True
    else:
        print("🟨 NOT DONE YET → " + fix_msg)
        return False

def banner(module, step, points):
    print("=" * 62)
    print(f"📤 MODULE {module} | {step} SUBMISSION BLOCK  ({points} pts)")
    print("=" * 62)
    print(f"Student:              {STUDENT_NAME}")
    print(f"Dataset:              {DATASET_CHOICE}")
    print(f"My research question: {MY_RESEARCH_QUESTION}")
    print(f"My stakeholder:       {MY_STAKEHOLDER}")
    print(f"Generated:            {datetime.datetime.now():%Y-%m-%d %H:%M}")
    print("-" * 62)
    _n = len(_CHECKS)
    _fail = [(fix, ok) for fix, (passed, ok) in _CHECKS.items() if not passed]
    if _n and not _fail:
        print(f"SELF-CHECK STATUS: ✅ ALL {_n} CHECKS PASSED")
    else:
        print(f"SELF-CHECK STATUS: 🟨 {len(_fail)} OF {_n} NOT DONE")
    for fix, (passed, ok) in _CHECKS.items():
        print(("   ✅ " + ok) if passed else ("   🟨 " + fix))
    print("-" * 62)

def next_week_card():
    """Prints the exact lines to paste into next week's project card."""
    g = globals()
    print("-" * 62)
    print("📋 YOUR PROJECT CARD FOR NEXT WEEK (copy these lines into Step 2 next time):")
    print(f'    STUDENT_NAME         = "{STUDENT_NAME}"')
    print(f'    DATASET_CHOICE       = "{DATASET_CHOICE}"')
    print(f'    MY_RESEARCH_QUESTION = "{MY_RESEARCH_QUESTION}"')
    print(f'    MY_STAKEHOLDER       = "{MY_STAKEHOLDER}"')
    if g.get("M04_COLUMN") and g.get("M04_FIX"):
        print(f'    M04_COLUMN = "{g["M04_COLUMN"]}"')
        print(f'    M04_FIX    = "{g["M04_FIX"]}"')
        if g.get("FLAG_FROM_COLUMN"):
            print(f'    FLAG_FROM_COLUMN = "{g["FLAG_FROM_COLUMN"]}"')
            print(f'    FLAG_ABOVE       = "{g.get("FLAG_ABOVE", "")}"')
            print(f'    FLAG_YES_LABEL   = "{g.get("FLAG_YES_LABEL", "")}"')
            print(f'    FLAG_NO_LABEL    = "{g.get("FLAG_NO_LABEL", "")}"')

# Where the four datasets live (your instructor hosts these; do not edit)
DATA_URLS = {
    "diabetes":     "https://raw.githubusercontent.com/ctg30/him4331-data/main/diabetic_data.csv",
    "hospitals":    "https://raw.githubusercontent.com/ctg30/him4331-data/main/cms_hospitals.csv",
    "readmissions": "https://raw.githubusercontent.com/ctg30/him4331-data/main/cms_readmissions.csv",
    "breaches":     "https://raw.githubusercontent.com/ctg30/him4331-data/main/hhs_breaches.csv",
}
_MISS_CODES = ["?", "Unknown", "UNKNOWN", "Not Available", "N/A", "NA", "None", ""]
print("✅ Step 1 done. Tools loaded. Go to Step 2.")

## 🟨 Step 2: your project card
Open last week's PDF, find the box titled 📋 YOUR PROJECT CARD FOR NEXT WEEK, and paste those lines over the blank ones below. Then press ▶. Your question is allowed to get sharper as you learn; if it changed, use the newer version.

The Module 4 lines re-apply the data fix you chose that week, so this notebook works on the same cleaned data. Your Module 4 PDF (and every PDF after it) prints them for you. If you never built a MY_FLAG column, leave those four blank.

In [ ]:
#@title 🟨 Step 2: fill in your project card
try:
    STUDENT_NAME         = ""   # e.g. "Jordan Rivera"
    DATASET_CHOICE       = ""   # one of: "diabetes", "hospitals", "readmissions", "breaches"
    MY_RESEARCH_QUESTION = ""   # e.g. "Do longer hospital stays predict 30-day readmission for diabetic patients?"
    MY_STAKEHOLDER       = ""   # who you're writing for, e.g. "Quality Director at a 300-bed hospital"


    # ---- carried over from Module 4 (copy from your M04 PDF) ----
    M04_COLUMN = ""   # the column you fixed in Module 4, e.g. "race"
    M04_FIX    = ""   # the fix you chose: recode_missing / drop_rows / impute_median / to_number / drop_duplicates / document_only

    # ---- OPTIONAL yes/no column MY_FLAG (only if you built one in Module 4; leave blank otherwise) ----
    FLAG_FROM_COLUMN = ""   # e.g. "Excess Readmission Ratio"
    FLAG_ABOVE       = ""   # e.g. "1.0"
    FLAG_YES_LABEL   = ""   # e.g. "Worse than expected"
    FLAG_NO_LABEL    = ""   # e.g. "As expected or better"

    # ---- self-checks (don't edit below) ----
    DATASET_CHOICE = DATASET_CHOICE.strip().lower()
    check(len(STUDENT_NAME.strip()) > 2, "Name entered.", "Type your name in STUDENT_NAME.")
    check(DATASET_CHOICE in DATA_URLS, f"Dataset '{DATASET_CHOICE}' is valid.",
          "DATASET_CHOICE must be exactly one of: diabetes, hospitals, readmissions, breaches")
    check(len(MY_RESEARCH_QUESTION.strip()) > 15 and MY_RESEARCH_QUESTION.strip().endswith("?"),
          "Research question entered.",
          "Write your research question as one full sentence ending in '?'")
    check(len(MY_STAKEHOLDER.strip()) > 5, "Stakeholder entered.",
          "Name a real decision-maker role, e.g. 'HIM privacy officer'.")

    _FIX_MENU = ["recode_missing", "drop_rows", "impute_median", "to_number", "drop_duplicates", "document_only"]
    M04_COLUMN, M04_FIX = M04_COLUMN.strip(), M04_FIX.strip().lower()
    FLAG_FROM_COLUMN, FLAG_ABOVE = FLAG_FROM_COLUMN.strip(), FLAG_ABOVE.strip()
    check(len(M04_COLUMN) > 0, "M04 column entered.", "Type the column you fixed in Module 4 into M04_COLUMN.")
    check(M04_FIX in _FIX_MENU, f"M04 fix '{M04_FIX}' is valid.",
          "M04_FIX must be exactly one of: " + ", ".join(_FIX_MENU))

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## ▶ Step 3: load your data (nothing to type, just press ▶)

In [ ]:
#@title ▶ Step 3: load and clean your dataset (just run it)
try:
    try:
        df = pd.read_csv(DATA_URLS[DATASET_CHOICE], low_memory=False)
        print(f"✅ Loaded '{DATASET_CHOICE}': {df.shape[0]:,} rows × {df.shape[1]} columns")
    except KeyError:
        df = None
        print("🟨 Fix DATASET_CHOICE in Step 2 first, then re-run this cell.")
    except Exception as e:
        df = None
        print("🟨 Could not load the data. Check your internet connection and re-run.")
        print("   If it still fails, screenshot this and message your instructor:", repr(e))

    if df is not None:
        # ---- re-apply the Module 4 fix ----
        df_clean = df.copy()
        if M04_COLUMN not in df.columns:
            print(f"🟨 '{M04_COLUMN}' is not a column in this dataset. Check M04_COLUMN in Step 2 (copy it from your M04 PDF).")
        elif M04_FIX == "recode_missing":
            df_clean[M04_COLUMN] = df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan)
        elif M04_FIX == "drop_rows":
            df_clean[M04_COLUMN] = df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan)
            df_clean = df_clean.dropna(subset=[M04_COLUMN])
        elif M04_FIX == "impute_median":
            df_clean[M04_COLUMN] = pd.to_numeric(df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan), errors="coerce")
            df_clean[M04_COLUMN] = df_clean[M04_COLUMN].fillna(df_clean[M04_COLUMN].median())
        elif M04_FIX == "to_number":
            df_clean[M04_COLUMN] = pd.to_numeric(df_clean[M04_COLUMN].replace(_MISS_CODES, np.nan), errors="coerce")
        elif M04_FIX == "drop_duplicates":
            df_clean = df_clean.drop_duplicates()
        print(f"✅ Module 4 fix re-applied: {M04_FIX} on '{M04_COLUMN}'. Rows: {len(df):,} → {len(df_clean):,}")

        # ---- optional MY_FLAG ----
        if FLAG_FROM_COLUMN == "" and FLAG_ABOVE == "":
            print("⏭️  No MY_FLAG settings entered (fine unless you used one in Module 4).")
        elif FLAG_FROM_COLUMN not in df_clean.columns:
            print(f"🟨 '{FLAG_FROM_COLUMN}' isn't a column in your dataset. Copy the exact name, or clear all four FLAG boxes.")
        else:
            try:
                _cut = float(FLAG_ABOVE)
            except ValueError:
                _cut = None
                print(f"🟨 FLAG_ABOVE ('{FLAG_ABOVE}') needs to be a number in quotes, like \"1.0\".")
            if _cut is not None:
                _num = pd.to_numeric(df_clean[FLAG_FROM_COLUMN], errors="coerce")
                _yes = FLAG_YES_LABEL.strip() or "Above cutoff"
                _no = FLAG_NO_LABEL.strip() or "At or below cutoff"
                _flag = pd.Series([None] * len(df_clean), index=df_clean.index, dtype=object)
                _flag[_num.notna() & (_num > _cut)] = _yes
                _flag[_num.notna() & (_num <= _cut)] = _no
                df_clean["MY_FLAG"] = _flag
                print(f"✅ MY_FLAG rebuilt from '{FLAG_FROM_COLUMN}' (cutoff {_cut}). Counts:")
                print(df_clean["MY_FLAG"].value_counts(dropna=False).to_string())
        print("✅ Step 3 done. Your cleaned data (df_clean) is ready.")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## 🟨 Step 4: this week's work
💡 **Pairings that work in YOUR file.** I've verified each one produces a meaningful table:

* **diabetes** → `age` × `readmitted` · `race` × `readmitted` · `insulin` × `readmitted` · `change` × `readmitted`
* **hospitals** → `Hospital Ownership` × `Hospital overall rating` · `Hospital Type` × `Emergency Services` · `State` × `Hospital overall rating`
* **readmissions** → `Measure Name` × `MY_FLAG` (needs the MY_FLAG settings in your project card; `State` × `Measure Name` won't work, because every hospital reports all six measures, so there's nothing to detect)
* **breaches** → `Covered Entity Type` × `Type of Breach` · `Business Associate Present` × `Type of Breach` · `Location of Breached Information` × `Covered Entity Type`

In [ ]:
#@title 🟨 TODO: hypothesis + your two categorical variables
try:
    M05_HYPOTHESIS = ""   # e.g. "Readmission status is associated with age group."
    CAT_VAR_A = ""        # exact column name (e.g. the group)
    CAT_VAR_B = ""        # exact column name (e.g. the outcome)

    CAT_VAR_A, CAT_VAR_B = CAT_VAR_A.strip(), CAT_VAR_B.strip()
    check(len(M05_HYPOTHESIS.strip()) > 10, "Hypothesis stated.", "Write M05_HYPOTHESIS as one sentence: 'X is associated with Y.'")
    check(CAT_VAR_A in df_clean.columns and CAT_VAR_B in df_clean.columns,
          "Both variables found.", "CAT_VAR_A / CAT_VAR_B must be exact column names (or MY_FLAG). Column list: " + str(df_clean.columns.tolist()))
    check(CAT_VAR_A != CAT_VAR_B, "Two different variables chosen.",
          "CAT_VAR_A and CAT_VAR_B must be two DIFFERENT columns.")
    for _v in (CAT_VAR_A, CAT_VAR_B):
        if _v in df_clean.columns:
            check(df_clean[_v].nunique() <= 60,
                  f"'{_v}' has {df_clean[_v].nunique()} categories (fine).",
                  f"'{_v}' has {df_clean[_v].nunique()} different values, which is too many for a readable table. "
                  "Pick a category column with a handful of groups, or use MY_FLAG.")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

In [ ]:
#@title ▶ Contingency table + chi-square test (just run it)
try:
    ct = pd.crosstab(df_clean[CAT_VAR_A], df_clean[CAT_VAR_B])
    print("CONTINGENCY TABLE:")
    print(ct.to_string())
    print()
    chi2, p_value_m05, dof, _ = stats.chi2_contingency(ct)
    print(f"chi-square = {chi2:.3f}   df = {dof}   p-value = {p_value_m05:.4f}")
    print()
    print("📖 How to read this: a p-value BELOW 0.05 = strong evidence the two variables")
    print("   are associated. A p-value ABOVE 0.05 = we can't rule out coincidence.")
    print("   (Tip: p = 0.0000 just means it is smaller than 0.0001. Write 'p < 0.0001'.)")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")
except Exception as e:
    print("🟨 Something unexpected happened in this cell. Try Runtime → Run all first.")
    print("   If it still fails, screenshot this and message your instructor:", repr(e))

### ✍️ What a good interpretation looks like
Flu-clinic example (their test gave p = 0.002):

> "Because the p-value (0.002) is well below 0.05, we have strong evidence that second-dose completion is genuinely associated with insurance type. This pattern is very unlikely to be coincidence. Uninsured patients completed their second dose far less often than any other group in the table. For a clinic operations manager, this means reminder calls and flexible scheduling should be aimed at uninsured patients first."

Pattern: **p-value in plain words → which groups actually differ (read your table!) → what the stakeholder should do.**

In [ ]:
#@title 🟨 TODO: your 3-sentence interpretation (finish the starters)
M05_INTERPRETATION = """
Because the p-value was [above / below] 0.05, we [do / do not] have strong evidence that ...
Looking at the table, the groups that stand out are ...
For my stakeholder, this means they should ...
"""

check("[above / below]" not in M05_INTERPRETATION
      and "[do / do not]" not in M05_INTERPRETATION
      and "stand out are ..." not in M05_INTERPRETATION
      and "they should ..." not in M05_INTERPRETATION,
      "Interpretation written.", "Finish all three sentence starters and replace every [bracketed choice] with your answer.")

## 📤 Step 5: submission block

In [ ]:
#@title 📤 M05.4 SUBMISSION BLOCK (Module 5): run, then File → Print → Save as PDF
try:
    banner(5, "M05.4 Categorical Inference", 15)
    print("Data prep carried over from M04:", M04_FIX, "on", repr(M04_COLUMN), "| MY_FLAG:", FLAG_FROM_COLUMN or "not used")
    print()
    print("1) HYPOTHESIS:", M05_HYPOTHESIS)
    print()
    print("2) CONTINGENCY TABLE:", f"({CAT_VAR_A} × {CAT_VAR_B})")
    print(ct.to_string())
    print()
    print(f"3) TEST OUTPUT: chi-square = {chi2:.3f}, df = {dof}")
    print(f"4) P-VALUE: p = {p_value_m05:.4f}")
    print()
    print("5) MY INTERPRETATION:")
    print(M05_INTERPRETATION.strip())
    next_week_card()
    print("=" * 62)
    print("📎 Now: File → Print → Save as PDF → upload to the M05.4 Canvas assignment (Module 5).")

except (KeyError, NameError, AttributeError, TypeError, IndexError):
    print("🟨 This cell needs an earlier cell finished first.")
    print("   Scroll up, complete the 🟨 TODO cells above in order, then re-run this cell.")
    print("   (Fastest fix: Runtime → Run all)")

## 📎 Step 5: export and submit
1. Make sure the submission block above printed with **no 🟨 warnings**.
2. `File → Print` (or Ctrl+P / Cmd+P) → choose **Save as PDF** → save it as `HIM4331_M05.4_LastnameFirstname.pdf`.
3. Upload that PDF to the Canvas assignment named **M05.4** (Module 5).
4. Keep this notebook. You will copy your project card from it next week.